# ApacheJIT: repository-derived commit labels

Evidence class: bug-inducing commit labels derived from repository history; these are not labels of observed runtime failures. The existing Isoprax verifier checks the supplied canonical local artifact. Charts show per-project commit volume and positive-label yield, including projects with zero positives. No cross-project or cross-family predictor score is inferred.

Expected local input: `data/apachejit_total.csv`, or set `ISOPRAX_DATA_DIR` before starting Jupyter.

In [ ]:
import os
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
working_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file() and (candidate / "isoprax").is_dir()
    ),
    None,
)
configured_root = os.environ.get("ISOPRAX_REPO_ROOT")
REPO_ROOT = (
    Path(configured_root).expanduser().resolve() if configured_root else working_root
)
if (
    REPO_ROOT is None
    or not (REPO_ROOT / "pyproject.toml").is_file()
    or not (REPO_ROOT / "isoprax").is_dir()
):
    raise RuntimeError(
        "Launch from this checkout or set ISOPRAX_REPO_ROOT to its repository root."
    )
if configured_root and working_root is not None and REPO_ROOT != working_root:
    raise RuntimeError(
        "ISOPRAX_REPO_ROOT does not match the notebook's checkout directory."
    )
repo_path = str(REPO_ROOT)
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0, repo_path)

import isoprax

PACKAGE_ROOT = Path(isoprax.__file__).resolve().parents[1]
if PACKAGE_ROOT != REPO_ROOT:
    raise RuntimeError(
        "The selected Python kernel does not import Isoprax from this checkout."
    )

from notebooks._support import (
    data_root_from_environment,
    display_review,
    review_dataset,
)

DATA_ROOT = data_root_from_environment(REPO_ROOT)
CSV_PATH = DATA_ROOT / "apachejit_total.csv"
print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Local CSV: {CSV_PATH}")

In [ ]:
review = review_dataset("apachejit", {"csv": CSV_PATH}, REPO_ROOT)
display_review(review)

A zero-yield project is a real observed category, not missing data. The temporal split uses parsed `author_date` in UTC: before 2017-01-01 for training and that instant or later for testing. The source `year` is audited but not trusted as the split key. Only the 12 commit metrics enter the prior-only and logistic-regression comparisons; project, identifiers, labels, and date fields are excluded. Results concern repository-derived buggy-commit labels, not runtime failures. Failed or unavailable verification gates all raw-row analysis.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from notebooks import _analysis

if review.verification.status in {"verified", "verified_with_warnings"}:
    frame = pd.read_csv(CSV_PATH)
    frame["buggy_label"] = frame["buggy"].map(
        {True: 1, False: 0, "True": 1, "False": 0}
    )
    if frame["buggy_label"].isna().any():
        raise ValueError("Verifier-approved buggy labels did not parse as booleans")
    frame["author_time_utc"] = pd.to_datetime(
        pd.to_numeric(frame["author_date"]), unit="s", utc=True
    )
    frame["author_year_utc"] = frame["author_time_utc"].dt.year

    display(Markdown("## Repository, label-yield, and timestamp audit"))
    project_yield = frame.groupby("project")["buggy_label"].agg(
        commits="size", positives="sum"
    )
    project_yield["positive_yield"] = (
        project_yield["positives"] / project_yield["commits"]
    )
    display(project_yield.sort_values("commits", ascending=False))
    frame["year_matches_author_date"] = (
        frame["year"].astype(int) == frame["author_year_utc"]
    )
    print(
        f"Rows where source 'year' differs from author_date's UTC year: {(~frame['year_matches_author_date']).sum():,} / {len(frame):,}"
    )
    print(
        "The source year is audited but is not used for temporal splitting or prediction."
    )

    annual = frame.groupby("author_year_utc")["buggy_label"].agg(
        commits="size", positives="sum"
    )
    annual["positive_yield"] = annual["positives"] / annual["commits"]
    display(annual)
    figure, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    project_yield[["commits", "positives"]].sort_values("commits").plot.barh(ax=axes[0])
    axes[0].set_title("Per-project commit and positive counts")
    axes[0].set_xlabel("Commits")
    annual["positive_yield"].plot(ax=axes[1], marker="o", color="#b84a4a")
    axes[1].set_title("Buggy-label yield by author_date UTC year")
    axes[1].set_ylabel("Positive yield")
    axes[1].set_xlabel("UTC year")
    display(figure)
    plt.close(figure)

    display(
        Markdown(
            "## Fixed chronological baseline: author_date before 2017-01-01 UTC → train"
        )
    )
    cutoff = pd.Timestamp("2017-01-01T00:00:00Z").timestamp()
    train_indices, test_indices = _analysis.timestamp_holdout_indices(
        pd.to_numeric(frame["author_date"]), cutoff=cutoff
    )
    y = frame["buggy_label"].astype(int)
    train_y, test_y = y.iloc[train_indices], y.iloc[test_indices]
    print(
        f"Train: {len(train_indices):,} commits with {int(train_y.sum()):,} positives; test: {len(test_indices):,} commits with {int(test_y.sum()):,} positives"
    )
    test_project_yield = (
        frame.iloc[test_indices]
        .assign(buggy_label=test_y.to_numpy())
        .groupby("project")["buggy_label"]
        .agg(commits="size", positives="sum")
    )
    test_project_yield["positive_yield"] = (
        test_project_yield["positives"] / test_project_yield["commits"]
    )
    display(test_project_yield.sort_values("commits", ascending=False))
    if train_y.nunique() < 2:
        display(
            Markdown(
                "Baseline not estimable: training partition contains only one outcome class."
            )
        )
    else:
        X = frame.loc[:, _analysis.APACHEJIT_MODEL_FEATURES]
        models = {
            "prior dummy": DummyClassifier(strategy="prior"),
            "logistic regression": make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler(),
                LogisticRegression(max_iter=1000, random_state=41),
            ),
        }
        metric_rows = []
        for name, model in models.items():
            model.fit(X.iloc[train_indices], train_y)
            probabilities = model.predict_proba(X.iloc[test_indices])[:, 1]
            metric_rows.append(
                {
                    "model": name,
                    **_analysis.binary_classification_metrics(test_y, probabilities),
                }
            )
        metrics = pd.DataFrame(metric_rows).set_index("model")
        display(metrics)
        figure, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        metrics["average_precision"].astype(float).plot.bar(ax=axes[0], color="#3977a8")
        axes[0].set_title("Average precision (test support shown above)")
        metrics["brier_score"].astype(float).plot.bar(ax=axes[1], color="#b87932")
        axes[1].set_title("Brier score (lower is better)")
        display(figure)
        plt.close(figure)
else:
    display(
        Markdown(
            "Exploration is gated: raw rows are read only after the canonical verifier succeeds."
        )
    )